In [2]:
import os, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input as pre_res
from tensorflow.keras.applications.efficientnet import preprocess_input as pre_eff
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import f1_score

BASE = '/kaggle/input/datasets/ekrasafdar/brain-tumor-mri'
TRAIN_DIR = os.path.join(BASE, 'Training')
TEST_DIR  = os.path.join(BASE, 'Testing')

def mc_model(name):
    inp = layers.Input(shape=(224, 224, 3))
    if name == 'resnet50':
        base = ResNet50(weights='imagenet', include_top=False, input_tensor=inp)
        pre = pre_res
    else:
        base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=inp)
        pre = pre_eff
    
    base.trainable = True
    for L in base.layers[:-50]: L.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.5)(x, training=True)  # MC Dropout
    out = layers.Dense(4, activation='softmax')(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return m, pre

# Train ResNet50 MC (seed 42 only, ~15 min)
for model_name in ['resnet50', 'efficientnetb0']:
    print(f"\n{'='*50}\nMC Dropout: {model_name.upper()}\n{'='*50}")
    model, pre = mc_model(model_name)
    
    tr_aug = ImageDataGenerator(preprocessing_function=pre, rotation_range=20,
        width_shift_range=0.15, height_shift_range=0.15, zoom_range=0.15,
        horizontal_flip=True, brightness_range=[0.8, 1.2], validation_split=0.2)
    tr = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
        class_mode='categorical', subset='training', seed=42)
    val = tr_aug.flow_from_directory(TRAIN_DIR, target_size=(224,224), batch_size=32,
        class_mode='categorical', subset='validation', shuffle=False, seed=42)
    te = ImageDataGenerator(preprocessing_function=pre).flow_from_directory(
        TEST_DIR, target_size=(224,224), batch_size=32, class_mode='categorical', shuffle=False)
    
    cb = [
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
    ]
    model.fit(tr, validation_data=val, epochs=30, callbacks=cb, verbose=1)
    
    te.reset(); loss, acc = model.evaluate(te, verbose=0)
    te.reset(); yp = np.argmax(model.predict(te, verbose=0), axis=1)
    yt = te.classes
    print(f"{model_name} MC — Accuracy: {acc*100:.2f}% | F1: {f1_score(yt, yp, average='weighted')*100:.2f}%")


MC Dropout: RESNET50


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/ekrasafdar/brain-tumor-mri/Training'